# 30. GraphRAG with a Milvus Vector DB

Store relations as vectors in Milvus, retrieve the closest, then expand through the graph.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Notebook 29 kept everything in memory. For real corpora you want a **vector database**. Here we store each *relation* (`hnsw is a graph index`) as a vector in **Milvus Lite** (a local, file-based Milvus), search the relations semantically, and then expand through the graph to pull in related facts. This combines semantic recall (vector DB) with structured reasoning (graph).

## How it works

1. Extract triples and build a networkx graph
2. Embed each relation sentence
3. Insert vectors + text into a Milvus collection
4. Search Milvus for relations similar to the question
5. Expand: add other edges touching the matched entities
6. Generate from the expanded facts

```text
 triples --> relation text --> embed --> Milvus collection
 question --> embed --> Milvus ANN search --> seed relations --> graph expansion --> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install pymilvus networkx`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 11:07:09.287000 9361 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
import networkx as nx
from pymilvus import MilvusClient

G = nx.Graph()
relations = []
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r)
        relations.append(f"{s} {r} {o}")
print(len(relations), "relations")

93 relations


Milvus Lite runs from a local file - no server required.

In [3]:
client = MilvusClient(str(DATA / "graph.db"))
if client.has_collection("relations"):
    client.drop_collection("relations")
client.create_collection(collection_name="relations", dimension=embed("x").shape[1])

vecs = embed(relations)
client.insert("relations", [{"id": i, "vector": vecs[i].tolist(), "text": relations[i]} for i in range(len(relations))])

ConnectionConfigException: <ConnectionConfigException: (code=1, message=milvus-lite is required for local database connections. Please install it with: pip install pymilvus[milvus_lite])>

In [4]:
def milvus_graph_search(query, k=5, hops=1):
    res = client.search("relations", data=[embed(query)[0].tolist()], limit=k, output_fields=["text"])[0]
    seeds = [r["entity"]["text"] for r in res]
    # expand: add other edges touching the entities of the seed relations
    entities = {n for n in G.nodes if any(n in s for s in seeds)}
    expanded = set(seeds)
    for n in entities:
        for nb in G.neighbors(n):
            expanded.add(f"{n} {G[n][nb]['relation']} {nb}")
    return seeds, sorted(expanded)[:25]

query = "How do vector databases speed up nearest neighbour search?"
seeds, context = milvus_graph_search(query)
print("seed relations:", seeds)
print(answer(query, context))

NameError: name 'client' is not defined

## Inspect the collection

In [5]:
print(client.get_collection_stats("relations"))
rows = client.query("relations", filter="id < 5", output_fields=["id", "text"])
for r in rows: print(r)

NameError: name 'client' is not defined

## Seed relations vs expanded context
How much does graph expansion add?

In [6]:
for q in ["How do vector databases speed up nearest neighbour search?", "What does BM25 score documents by?"]:
    seeds, expanded = milvus_graph_search(q)
    print(f"{q}\n   seeds: {len(seeds)}   after expansion: {len(expanded)}")
    for s in seeds[:3]: print("     seed:", s)

NameError: name 'client' is not defined

## Compare against vector RAG over raw chunks

In [7]:
compare_pipelines({"vector RAG": lambda q: rag(q, store, 3)[0], "Milvus + graph": lambda q: answer(q, milvus_graph_search(q)[1])}, n=4)

,keyword_recall,judge_correct,avg_seconds
pipeline,,,
vector RAG,0.79,1.0,0.0
Milvus + graph,0.00,0.0,0.0


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Persistent, scalable vector storage | More moving parts (DB + graph) |
| Semantic retrieval over relations | Extraction cost and noise as in GraphRAG |
| Graph expansion adds multi-hop context | Milvus Lite is for dev; use a Milvus server in production |

## When to use it

Graph-style knowledge that is too large for memory or needs to be shared across services.

## Experiments to try

- Also index entities in a second collection.
- Store `chunk_id` in Milvus and return source passages.
- Use a metric type / index (HNSW) explicitly.

## Key takeaways

- Vector DB for recall, graph for structure.
- Milvus Lite needs no server.
- Always close the loop by evaluating.

---
**Next:** [31. Microsoft GraphRAG (Global Search)](../31_Microsoft_GraphRag/Microsoft_GraphRag.ipynb)